In [40]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go
# If you want to import all functions from benchmark-utils.py, use the following:
import sys
sys.path.append('../utils')
from benchmark_utils import *

In [41]:
folders = ["../../../out/timespent_20251001_215650/"]

folder=folders[0]


df:pd.DataFrame = merge_benchmark_results(folders)
df = assign_cluster_and_role(df)
df["cpu"] = clean_dool_data(df).Dool.apply(extract_max_cpu_usage)


In [42]:
clients = df.query("Role == 0").groupby("folder")[["Throughput"]].sum().reset_index().sort_values("folder")

fig = go.Figure() 
fig.add_trace(go.Bar(x=clients.folder,y=clients.Throughput.astype(float)))
fig.show()

In [43]:
def plot_max_cpu_by_folder(role):
    folder_cpu_max = df.query("Role == @role ").groupby("folder")[["cpu"]].max().reset_index().sort_values("folder")
    fig = go.Figure() 
    fig.add_trace(go.Bar(x=folder_cpu_max.folder,y=folder_cpu_max.cpu.astype(float)))
    fig.show()

plot_max_cpu_by_folder(2)
plot_max_cpu_by_folder(1)

In [44]:
def generate_pivot_tables_with_bytes(df):
    def generate_bytes_pivot_table(r):
        if r["Role"] == 0:
            return pd.DataFrame()
        df_bytes = pd.DataFrame(r["BytesSentTo"])
        df_bytes["ToRole"] = df_bytes["CID"].apply(lambda cid: r["IDtoRole"][cid])
        pivot = df_bytes.pivot_table(columns='Typ', values='Value', aggfunc='sum')
        pivot['index'] = r.name  # Use row index for merging
        return pivot.reset_index()

# Collect all pivot tables
    pivot_tables = [generate_bytes_pivot_table(row) for _, row in df.iterrows() if row["Role"] != 0]
    pivot_df = pd.concat(pivot_tables, ignore_index=True).fillna(0)

# Merge with original df on index
    df_with_bytes = df.merge(pivot_df, left_index=True, right_on='index', how='left')
    return df_with_bytes

df_with_bytes = generate_pivot_tables_with_bytes(df)


In [45]:
types = ["Checkpoint",	"ClientResponse"		,"Commit"	,"PrePrepare"	,"Prepare"]

average_by_folder_and_role = df_with_bytes.groupby(["folder","Role"]).mean(numeric_only=True).reset_index()
total_bytes_sent = average_by_folder_and_role[["folder","Role"]+types].groupby(["folder","Role"]).sum().sum(axis=1).reset_index().set_index(["folder","Role"])
total_bytes_sent[0].index.set_names(['folder', 'Role'], inplace=True)

normalized_bytes_per_folder = (
    average_by_folder_and_role.set_index(["folder", "Role"])[types]
    .div(total_bytes_sent[0], axis=0)
    .reset_index()
)
normalized_bytes_per_folder["Role"]=average_by_folder_and_role["Role"]
fig = go.Figure()

for typ in types:
    fig.add_trace(go.Bar(x=normalized_bytes_per_folder.Role.apply(str)+normalized_bytes_per_folder.folder.apply(str),y=normalized_bytes_per_folder[typ],name=typ))
fig.update_layout(barmode='stack')
fig.show()

In [46]:
total_bytes_sent[0]/df.groupby(["folder","Role"])[["RunTime"]].mean().RunTime*8*10**(-9)

folder                                   Role
../../../out/timespent_20251001_215650/  0       0.000000
                                         1       0.064678
                                         2       1.397856
dtype: float64

In [66]:
def generate_pivot_tables_with_timespent(df):
    def generate_bytes_pivot_table(r):
        if r["Role"] == 0:
            return pd.DataFrame()
        df_timespent = pd.DataFrame(r["TimeSpent"])
        df_timespent["Stage"] = df_timespent["Typ"].apply(lambda s : s.split(":")[1])
        df_timespent["Typ"] = df_timespent["Typ"].apply(lambda s : s.split(":")[0])
        # df_timespent["ToRole"] = df_timespent["CID"].apply(lambda cid: r["IDtoRole"][cid])
        pivot = df_timespent.pivot_table(columns=['Typ'],index="Stage", values='Value', aggfunc='sum')
        pivot['index'] = r.name  # Use row index for merging
        return pivot.reset_index()

# Collect all pivot tables
    pivot_tables = [generate_bytes_pivot_table(row) for _, row in df.iterrows() if row["Role"] != 0]
    pivot_df = pd.concat(pivot_tables, ignore_index=True).fillna(0)
    # display(pivot_df)

# Merge with original df on index
    df_with_timespent = df.merge(pivot_df, left_index=True, right_on='index', how='left')
    return pivot_df.drop(columns=["index","Stage"]).columns.to_list(),df_with_timespent

types , df_with_timespent = generate_pivot_tables_with_timespent(df)
# df
types

['CreateSigForHash-MAC',
 'MsgType_Checkpoint',
 'MsgType_Commit',
 'MsgType_Prepare',
 'ReadFull',
 'computeMsgHash',
 'conn.Write',
 'handleClientRequest',
 'handleValidPrePrepare',
 'proto.Marshal',
 'proto.Unmarshal',
 'verifyMACSig',
 'verifyPKSig']

In [75]:
# types = ["Checkpoint",	"ClientResponse"		,"Commit"	,"PrePrepare"	,"Prepare"]

average_by_folder_and_role = df_with_timespent.groupby(["folder","Role","Stage"]).mean(numeric_only=True).reset_index()
total_bytes_sent = average_by_folder_and_role[["folder","Role","Stage"]+types].groupby(["folder","Role","Stage"]).sum().sum(axis=1).reset_index().set_index(["folder","Role","Stage"])
total_bytes_sent[0].index.set_names(['folder', 'Role',"Stage"], inplace=True)

normalized_bytes_per_folder = (
    average_by_folder_and_role.set_index(["folder", "Role","Stage"])[types]
    .div(total_bytes_sent[0], axis=0)
    .reset_index()
)
normalized_bytes_per_folder["Role"]=average_by_folder_and_role["Role"]
fig = go.Figure()

for typ in types:
    fig.add_trace(go.Bar(x=normalized_bytes_per_folder.Role.apply(str)+average_by_folder_and_role.Stage.apply(str)+normalized_bytes_per_folder.folder.apply(str),y=normalized_bytes_per_folder[typ],name=typ))
fig.update_layout(barmode='stack')
fig.show()

from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=6, shared_yaxes=True)

j = 1
for i,d in normalized_bytes_per_folder.groupby(["Stage"]):
    for typ in types:
        fig.add_trace(go.Bar(x=d.Role.apply(str)+d.folder.apply(str),y=d[typ],name=typ),1,j)
    j+=1 

fig.update_layout(barmode='stack')
fig.show()



In [84]:
def generate_pivot_tables_with_timespent(df):
    def generate_bytes_pivot_table(r):
        if r["Role"] == 0:
            return pd.DataFrame()
        df_timespent = pd.DataFrame(r["TimeSpent"])
        # df_timespent["Stage"] = df_timespent["Typ"].apply(lambda s : s.split(":")[1])
        # df_timespent["Typ"] = df_timespent["Typ"].apply(lambda s : s.split(":")[0])
        # df_timespent["ToRole"] = df_timespent["CID"].apply(lambda cid: r["IDtoRole"][cid])
        pivot = df_timespent.pivot_table(columns=['Typ'], values='Value', aggfunc='sum')
        pivot['index'] = r.name  # Use row index for merging
        return pivot.reset_index(drop=True)

# Collect all pivot tables
    pivot_tables = [generate_bytes_pivot_table(row) for _, row in df.iterrows() if row["Role"] != 0]
    pivot_df = pd.concat(pivot_tables, ignore_index=True).fillna(0)
    # display(pivot_df)

# Merge with original df on index
    df_with_timespent = df.merge(pivot_df, left_index=True, right_on='index', how='left')
    return pivot_df.drop(columns=["index"]).columns.to_list(),df_with_timespent

types , df_with_timespent = generate_pivot_tables_with_timespent(df)


average_by_folder_and_role = df_with_timespent.groupby(["folder","Role"]).mean(numeric_only=True).reset_index()
total_bytes_sent = average_by_folder_and_role[["folder","Role"]+types].groupby(["folder","Role"]).sum().sum(axis=1).reset_index().set_index(["folder","Role"])
total_bytes_sent[0].index.set_names(['folder', 'Role'], inplace=True)

normalized_bytes_per_folder = (
    average_by_folder_and_role.set_index(["folder", "Role"])[types]
    .div(total_bytes_sent[0], axis=0)
    .reset_index()
)

normalized_bytes_per_folder["Role"]=average_by_folder_and_role["Role"]

fig = go.Figure()


for typ in types:
    stage = typ.split(":")[1]
    
    fig.add_trace(go.Bar(x=stage+normalized_bytes_per_folder.Role.apply(str),y=normalized_bytes_per_folder[typ],name=typ.split(":")[0]))
fig.update_layout(barmode='stack')
fig.show()
# df

In [49]:
import subprocess
import requests
import time
from IPython.display import IFrame
import signal
port = 8010
for folder in reversed(folders):
    port +=1 
    try: 
        p = subprocess.Popen(["go","tool","pprof","-no_browser","-http",f"localhost:{port}",os.path.join(folder,"cli"), os.path.join(folder,"cpu.prof")] ,start_new_session=True)
        time.sleep(4)
   
    
        print(folder)

        display(IFrame(src=f"http://localhost:{port}/ui/flamegraph", width="100%", height=600))

       
        p.wait(timeout=2)
    except subprocess.TimeoutExpired: 
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except Exception as e :
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        raise e 
        



Serving web UI on http://localhost:8011


../../../out/timespent_20251001_215650/


go tool pprof: signal: terminated
